<a href="https://colab.research.google.com/github/ys23-lys/ESAA/blob/main/ESAA_OB2_%ED%94%84%EB%A1%9C%EC%A0%9D%ED%8A%B82_%EC%9D%B4%EC%97%B0%EC%88%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
import re

In [10]:
#파일 불러오기
train=pd.read_csv('/content/drive/MyDrive/Colab/ESAA/novelwriter/train.csv',encoding='utf-8')
test=pd.read_csv('/content/drive/MyDrive/Colab/ESAA/novelwriter/test_x.csv',encoding='utf-8')
sample_submission=pd.read_csv('/content/drive/MyDrive/Colab/ESAA/novelwriter/sample_submission.csv',encoding='utf-8')

In [11]:
train

,index,text,author
0,0,"He was almost choking. There was so much, so m...",3
1,1,"“Your sister asked for it, I suppose?”",2
2,2,"She was engaged one day as she walked, in per...",1
3,3,"The captain was in the porch, keeping himself ...",4
4,4,"“Have mercy, gentlemen!” odin flung up his han...",3
...,...,...,...
54874,54874,"“Is that you, Mr. Smith?” odin whispered. “I h...",2
54875,54875,"I told my plan to the captain, and between us ...",4
54876,54876,"""Your sincere well-wisher, friend, and sister...",1
54877,54877,“Then you wanted me to lend you money?”,3


In [12]:
test

,index,text
0,0,“Not at all. I think she is one of the most ch...
1,1,"""No,"" replied he, with sudden consciousness, ""..."
2,2,As the lady had stated her intention of scream...
3,3,“And then suddenly in the silence I heard a so...
4,4,His conviction remained unchanged. So far as I...
...,...,...
19612,19612,"At the end of another day or two, odin growing..."
19613,19613,"All afternoon we sat together, mostly in silen..."
19614,19614,"odin, having carried his thanks to odin, proc..."
19615,19615,"Soon after this, upon odin's leaving the room,..."


In [13]:
sample_submission

,index,0,1,2,3,4
0,0,0,0,0,0,0
1,1,0,0,0,0,0
2,2,0,0,0,0,0
3,3,0,0,0,0,0
4,4,0,0,0,0,0
...,...,...,...,...,...,...
19612,19612,0,0,0,0,0
19613,19613,0,0,0,0,0
19614,19614,0,0,0,0,0
19615,19615,0,0,0,0,0


# **전처리**

In [14]:
#부호를 제거해주는 함수
def alpha_num(text):
    return re.sub(r'[^A-Za-z0-9 ]','',text)

train['text']=train['text'].apply(alpha_num)

In [15]:
train

,index,text,author
0,0,He was almost choking There was so much so muc...,3
1,1,Your sister asked for it I suppose,2
2,2,She was engaged one day as she walked in peru...,1
3,3,The captain was in the porch keeping himself c...,4
4,4,Have mercy gentlemen odin flung up his hands D...,3
...,...,...,...
54874,54874,Is that you Mr Smith odin whispered I hardly d...,2
54875,54875,I told my plan to the captain and between us w...,4
54876,54876,Your sincere wellwisher friend and sister LUC...,1
54877,54877,Then you wanted me to lend you money,3


In [16]:
# 불용어 제거해주는 함수
def remove_stopwords(text):
    final_text = []
    for i in text.split():
        if i.strip().lower() not in stopwords:
            final_text.append(i.strip())
    return " ".join(final_text)

# 불용어
stopwords = [ "a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as",
             "at", "be", "because", "been", "before", "being", "below", "between", "both", "but", "by", "could",
             "did", "do", "does", "doing", "down", "during", "each", "few", "for", "from", "further", "had", "has",
             "have", "having", "he", "he'd", "he'll", "he's", "her", "here", "here's", "hers", "herself", "him", "himself",
             "his", "how", "how's", "i", "i'd", "i'll", "i'm", "i've", "if", "in", "into", "is", "it", "it's", "its", "itself",
             "let's", "me", "more", "most", "my", "myself", "nor", "of", "on", "once", "only", "or", "other", "ought", "our", "ours",
             "ourselves", "out", "over", "own", "same", "she", "she'd", "she'll", "she's", "should", "so", "some", "such", "than", "that",
             "that's", "the", "their", "theirs", "them", "themselves", "then", "there", "there's", "these", "they", "they'd", "they'll",
             "they're", "they've", "this", "those", "through", "to", "too", "under", "until", "up", "very", "was", "we", "we'd", "we'll",
             "we're", "we've", "were", "what", "what's", "when", "when's", "where", "where's", "which", "while", "who", "who's", "whom",
             "why", "why's", "with", "would", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself", "yourselves" ]

In [17]:
# 소문자 변환
def to_lowercase(text:str) -> str:
  """모든 알파벳을 소문자로 변환."""
  return text.lower()

In [18]:
# 숫자 토큰 치환
def replace_numbers(text:str) -> str:
    """
    숫자를 <NUM> 토큰으로 치환.
    - 숫자 값 자체보다 '숫자를 쓰는 패턴'이 문체 지표가 됨
    - 연속 숫자(예: 1, 2, 3)도 하나의 <NUM>으로 처리
    """
    text=re.sub(r'\b\d+(\.\d+)?\b','<NUM>',text)
    return text

In [19]:
# 반복 공백/개행 정리
def normalize_whitespace(text:str) -> str:
    """
    여러 공백 → 단일 공백, 여러 개행 → 단일 개행으로 정리.
    """
    # 여러 공백 → 단일 공백
    text=re.sub(r'[ \t]+',' ',text)
    # 세 개 이상 개행 → 두 개 개행 (단락 구분 보존)
    text=re.sub(r'\n{3,}','\n\n',text)
    return text.strip()

In [20]:
# 문장 분리 및 길이 통계 추출
def extract_sentence_stats(text: str) -> dict:
    """
    문장 분리 후 길이 분포 통계를 추출.
    문장 길이 분포는 ML 분류의 핵심 feature가 됨.

    반환값:
        {
            "sentences": [...],
            "num_sentences": int,
            "avg_sent_len_words": float,
            "avg_sent_len_chars": float,
            "sent_len_distribution": [int, ...]  # 각 문장의 단어 수
        }
    """
    sentences=sent_tokenize(text)
    word_counts=[len(s.split()) for s in sentences]
    char_counts=[len(s) for s in sentences]

    return {
        "sentences": sentences,
        "num_sentences": len(sentences),
        "avg_sent_len_words": round(sum(word_counts) / len(word_counts), 2) if word_counts else 0,
        "avg_sent_len_chars": round(sum(char_counts) / len(char_counts), 2) if char_counts else 0,
        "sent_len_distribution": word_counts,
    }

In [21]:
# 전처리 적용
train['text']=train['text'].apply(to_lowercase)
train['text']=train['text'].apply(replace_numbers)
train['text']=train['text'].apply(normalize_whitespace)

In [22]:
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # 추가된 부분

# 문장 길이 통계
train['sentence_stats']=train['text'].apply(extract_sentence_stats)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [23]:
train

,index,text,author,sentence_stats
0,0,he was almost choking there was so much so muc...,3,{'sentences': ['he was almost choking there wa...
1,1,your sister asked for it i suppose,2,{'sentences': ['your sister asked for it i sup...
2,2,she was engaged one day as she walked in perus...,1,{'sentences': ['she was engaged one day as she...
3,3,the captain was in the porch keeping himself c...,4,{'sentences': ['the captain was in the porch k...
4,4,have mercy gentlemen odin flung up his hands d...,3,{'sentences': ['have mercy gentlemen odin flun...
...,...,...,...,...
54874,54874,is that you mr smith odin whispered i hardly d...,2,{'sentences': ['is that you mr smith odin whis...
54875,54875,i told my plan to the captain and between us w...,4,{'sentences': ['i told my plan to the captain ...
54876,54876,your sincere wellwisher friend and sister lucy...,1,{'sentences': ['your sincere wellwisher friend...
54877,54877,then you wanted me to lend you money,3,{'sentences': ['then you wanted me to lend you...


# **모델링**

In [24]:
#파라미터 설정
vocab_size = 20000
embedding_dim = 16
max_length = 500
padding_type='post'
#oov_tok = "<OOV>"

In [25]:
#tokenizer에 fit
X_train = train['text']
y_train = train['author']

tokenizer = Tokenizer(num_words = vocab_size)#, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)
word_index = tokenizer.word_index

In [26]:
#데이터를 sequence로 변환해주고 padding 해줍니다.
train_sequences = tokenizer.texts_to_sequences(X_train)
train_padded = pad_sequences(train_sequences, padding=padding_type, maxlen=max_length)
X_test = test['text']
test_sequences = tokenizer.texts_to_sequences(X_test)
test_padded = pad_sequences(test_sequences, padding=padding_type, maxlen=max_length)

In [27]:
#가벼운 NLP모델 생성
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(5, activation='softmax')
])

In [28]:
# compile model
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# model summary
print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [29]:
# fit model
num_epochs = 20
history = model.fit(train_padded, y_train,
                    epochs=num_epochs, verbose=2,
                    validation_split=0.2)

Epoch 1/20
1372/1372 - 15s - 11ms/step - accuracy: 0.2747 - loss: 1.5661 - val_accuracy: 0.2707 - val_loss: 1.5538
Epoch 2/20
1372/1372 - 20s - 14ms/step - accuracy: 0.3081 - loss: 1.5186 - val_accuracy: 0.3389 - val_loss: 1.5020
Epoch 3/20
1372/1372 - 14s - 10ms/step - accuracy: 0.3661 - loss: 1.4275 - val_accuracy: 0.3679 - val_loss: 1.4061
Epoch 4/20
1372/1372 - 13s - 9ms/step - accuracy: 0.4191 - loss: 1.3444 - val_accuracy: 0.3877 - val_loss: 1.3334
Epoch 5/20
1372/1372 - 14s - 10ms/step - accuracy: 0.4669 - loss: 1.2701 - val_accuracy: 0.4910 - val_loss: 1.2183
Epoch 6/20
1372/1372 - 21s - 15ms/step - accuracy: 0.5005 - loss: 1.2057 - val_accuracy: 0.5248 - val_loss: 1.1616
Epoch 7/20
1372/1372 - 14s - 10ms/step - accuracy: 0.5238 - loss: 1.1636 - val_accuracy: 0.5263 - val_loss: 1.1732
Epoch 8/20
1372/1372 - 14s - 10ms/step - accuracy: 0.5476 - loss: 1.1126 - val_accuracy: 0.5897 - val_loss: 1.0638
Epoch 9/20
1372/1372 - 14s - 10ms/step - accuracy: 0.5683 - loss: 1.0700 - val_ac

In [30]:
# predict values
pred = model.predict(test_padded)

614/614 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [31]:
pred

array([[1.5439200e-03, 7.9140240e-01, 1.5021367e-01, 5.6181967e-02,
        6.5806258e-04],
       [3.3141050e-01, 2.7356827e-01, 2.3111844e-01, 2.3782546e-02,
        1.4012013e-01],
       [8.0319363e-01, 6.8748206e-02, 2.5172770e-02, 1.0664383e-03,
        1.0181894e-01],
       ...,
       [1.4369242e-02, 9.7823101e-01, 2.1922035e-05, 7.3701832e-03,
        7.6821152e-06],
       [2.2856472e-03, 9.8596609e-01, 3.9093010e-03, 7.2610914e-03,
        5.7789276e-04],
       [5.4251802e-01, 6.7763997e-04, 7.9836259e-03, 1.9208223e-04,
        4.4862866e-01]], dtype=float32)

In [32]:
# submission
sample_submission[['0','1','2','3','4']] = pred
sample_submission

,index,0,1,2,3,4
0,0,0.001544,0.791402,1.502137e-01,5.618197e-02,6.580626e-04
1,1,0.331410,0.273568,2.311184e-01,2.378255e-02,1.401201e-01
2,2,0.803194,0.068748,2.517277e-02,1.066438e-03,1.018189e-01
3,3,0.002173,0.000004,9.521347e-01,6.708892e-07,4.568710e-02
4,4,0.948106,0.031959,1.383636e-03,1.681340e-02,1.738005e-03
...,...,...,...,...,...,...
19612,19612,0.000003,0.999997,2.101721e-12,7.147955e-08,1.429910e-12
19613,19613,0.201217,0.000437,9.962764e-02,1.943540e-05,6.986992e-01
19614,19614,0.014369,0.978231,2.192203e-05,7.370183e-03,7.682115e-06
19615,19615,0.002286,0.985966,3.909301e-03,7.261091e-03,5.778928e-04


In [33]:
sample_submission.to_csv('submission.csv', index = False, encoding = 'utf-8')